In [1]:
import re

import pandas as pd
import requests

URL = "https://courselistings.wpi.edu/assets/prod-data.json"
# splits "CS 1101 - Introduction to Program Design" into code + title
TITLE_RE = re.compile(r"^(?P<code>[A-Za-z]+ ?\d+[A-Za-z]*) - (?P<title>.*)$")
HTML_TAG_RE = re.compile(r"<[^>]+>")

In [2]:
# pull raw course section entries (one per offered section, not per unique course)
entries = requests.get(URL, timeout=30).json()["Report_Entry"]
print(f"Pulled {len(entries)} course section entries from WPI course listings")

Pulled 3768 course section entries from WPI course listings


In [3]:
rows = []
for e in entries:
    m = TITLE_RE.match(e["Course_Title"])
    credits = float(e["Credits"])
    level = e["Academic_Level"]
    # grad credits are worth 3/2x their equivalent in UG credits (e.g. 3 grad credits = 4.5 UG credits)
    if level == "Graduate":
        credits_ug, credits_graduate = credits * 3 / 2, credits
    else:
        credits_ug, credits_graduate = credits, credits * 2 / 3
    rows.append(
        {
            "course_code": m["code"],
            "title": m["title"],
            "description": HTML_TAG_RE.sub("", e["Course_Description"]),
            "subject": e["Subject"],
            "credits_raw": credits,
            "credits_ug": credits_ug,
            "credits_graduate": credits_graduate,
            "academic_level": level,
            "department": e["Course_Section_Owner"],
            "start_date": e["Course_Section_Start_Date"],
        }
    )

courses_df = pd.DataFrame(rows).sort_values("start_date")
# collapse sections down to one row per course, keeping the most recent offering
courses_df = courses_df.drop_duplicates("course_code", keep="last")
courses_df = courses_df.set_index("course_code")

print(len(courses_df), "unique courses")
courses_df.head()

1282 unique courses


,title,description,subject,credits_raw,credits_ug,credits_graduate,academic_level,department,start_date
course_code,,,,,,,,,
AB 1531,Elementary Arabic I,Cat. IAn intensive course to introduce the Ara...,Arabic,3.0,3.0,2.0,Undergraduate,Humanities and Arts Department,2026-08-20
CS 557,Software Security Design And Analysis,Software is responsible for enforcing many cen...,Computer Science,3.0,4.5,3.0,Graduate,Computer Science Department,2026-08-20
ME 500,Applied Analytical Methods in Engineering,The emphasis of this course is on the modeling...,Mechanical Engineering,3.0,4.5,3.0,Graduate,Mechanical and Materials Engineering Department,2026-08-20
CS 555,Responsible Artificial Intelligence,CS 555 / DS 555: Responsible Artificial Intell...,Computer Science; Data Science,3.0,4.5,3.0,Graduate,Computer Science Department,2026-08-20
CS 554,Natural Language Processing,CS 554 / DS 554: Natural Language Processing (...,Computer Science; Data Science,3.0,4.5,3.0,Graduate,Computer Science Department,2026-08-20


In [4]:
courses_df.to_csv("courses.csv")